# Demo: Retrieval y Sistema Agéntico

Este notebook demuestra las diferentes estrategias de retrieval y el sistema agéntico.

In [1]:
import sys
sys.path.append('..')
from dotenv import load_dotenv
load_dotenv("../env/.env")

from graphrag.graph.neo4j_manager import Neo4jManager
from graphrag.retrieval.vector_retriever import VectorRetriever
from graphrag.retrieval.hybrid_retriever import HybridRetriever
from graphrag.retrieval.fulltext_retriever import FullTextRetriever
from graphrag.retrieval.text2cypher import Text2CypherRetriever
from graphrag.agents import AgenticRAG

/home/Pablo/Universidad/02-segundo-cuatrimestre/IAC/GraphRAG-IAC/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Inicializar conexión

In [2]:
neo4j = Neo4jManager()

## 2. Probar Vector Retrieval

In [3]:
vector_retriever = VectorRetriever(neo4j)

results = vector_retriever.retrieve("What is the average lifespan of a lion?")

print("Vector Search Results:")
for i, result in enumerate(results, 1):
    print(f"\n{i}. Score: {result['score']:.3f}")
    print(f"   {result['text']}")
    print(f"   {result['matched_questions']})")

ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download

## 3. Probar Full-text Retrieval

In [4]:
fulltext_retriever = FullTextRetriever(neo4j)

results = fulltext_retriever.retrieve("What is the average lifespan of an iguana?")

print("Full text Search Results:")
for i, result in enumerate(results, 1):
    print(f"\n{i}. Score: {result['score']:.3f}")
    print(f"   {result['text']}")

Full text Search Results:

1. Score: 5.196
   Animal: Iguana
Section: Reproduction, Babies, and Lifespan
These animals can live for 15 to 20 years and even longer if cared for properly. At the same time, the San Diego Zoo indicates that some of them can live as long as 60 years. However, wild iguanas have an average lifespan of eight years.

2. Score: 3.158
   Animal: Woodpecker
Section: Reproduction, Babies, and Lifespan
Once a baby first hatches, it develops quickly and is ready to leave the nest in about 30 days. On average, woodpeckers live between four and 12 years. Some can live up to 30 years if environmental conditions are just right.

3. Score: 3.012
   Animal: Bald Eagle
Section: Reproduction, Young, and Lifespan
It takes an average of 35 days for the chicks to emerge from their eggs sporting a brownish-gray head and tail mottled with white. In the 8 to 14 weeks it takes to gain their full-flight feathers, the juveniles spend a lot of time playing with each other, stretching 

## 4. Probar Hybrid Search Retrieval

In [3]:
hybrid_retriever = HybridRetriever(neo4j)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 12877.22it/s]


In [5]:
results = hybrid_retriever.retrieve("What is the average lifespan of an iguana?")

print("Hybrid Search Results:")
for i, result in enumerate(results, 1):
    print(f"\n{i}. Score: {result['score']:.3f}")
    print(f"   {result['text'][:200]}...")

Hybrid Search Results:

1. Score: 0.997
   Animal: Iguana
Section: Reproduction, Babies, and Lifespan
These animals can live for 15 to 20 years and even longer if cared for properly. At the same time, the San Diego Zoo indicates that some of t...

2. Score: 0.454
   Animal: Iguana
Section: Reproduction, Babies, and Lifespan
Females can store sperm from previous mates for several years if they cannot find suitable mates when ready to lay eggs.  
Females lay eggs f...

3. Score: 0.023
   Animal: Iguana
Section: Appearance
The different species have various sizes ranging from five to seven feet in length. Their whiplike tails make up about half of their body length. The desert iguana i...

4. Score: 0.016
   Animal: Iguana
Section: Habitat
Green iguanas are arboreal, living at the top ends of forest trees. Juveniles reside lower while mature iguanas live higher up. Living high in the tree canopy allows th...

5. Score: 0.016
   Animal: Iguana
Section: Population
You won’t find an overall 

## 5. Probar Text2Cypher

In [4]:
text2cypher = Text2CypherRetriever(neo4j)

In [5]:
# Añadir ejemplos few-shot
text2cypher.add_few_shot_example(
    "How many Mammal species are there?",
    "MATCH (s:Species)-[:BELONGS_TO_CLASS]->(c:AnimalClass {type: 'Mammal'}) RETURN count(s) AS totalMammals"
)

# Example A: Direct property retrieval (Factual)
text2cypher.add_few_shot_example(
    "What is the top speed and maximum weight of a Lion?",
    "MATCH (s:Species {name: 'Lion'}) RETURN s.top_speed_kmh, s.weight_max_kg"
)

# Example B: One-hop traversal
text2cypher.add_few_shot_example(
    "What type of diet do tigers have and what do they feed on?",
    "MATCH (s:Species {name: 'Tiger'})-[:HAS_DIET_TYPE]->(d:DietType), (s)-[:FEEDS_ON]->(f:FoodSource) RETURN d.type, f.type"
)

# Example C: Numerical property filtering
text2cypher.add_few_shot_example(
    "Which animals have a lifespan greater than 50 years?",
    "MATCH (s:Species) WHERE s.lifespan_years > 50 RETURN s.name, s.lifespan_years"
)

# Example D: Relationships with properties (Migration)
text2cypher.add_few_shot_example(
    "Where do whales migrate to in the winter?",
    "MATCH (s:Species {name: 'Whale'})-[m:MIGRATES_TO {season: 'winter'}]->(l:Location) RETURN l.type"
)

# Example E: Graph RAG Integration (Entities to Documents)
text2cypher.add_few_shot_example(
    "In which documents are carnivorous animals mentioned?",
    "MATCH (d:DietType {type: 'Carnivore'})<-[:HAS_DIET_TYPE]-(s:Species)<-[:HAS_ENTITY]-(c:Chunk)<-[:HAS_CHUNK]-(doc:Document) RETURN DISTINCT doc.title"
)

# Ejemplo para Carnívoros (PREYS_ON)
text2cypher.add_few_shot_example(
    "What animals do wolves hunt?",
    "MATCH (s:Species {name: 'Wolf'})-[:PREYS_ON]->(prey:Species) RETURN prey.name"
)

text2cypher.add_few_shot_example(
    "What do cows eat?",
    "MATCH (s:Species {name: 'Cow'})-[:FEEDS_ON]->(f:FoodSource) RETURN f.type"
)

# Ejemplo para Herbívoros (FEEDS_ON)
text2cypher.add_few_shot_example(
    "What kind of plants do elephants feed on?",
    "MATCH (s:Species {name: 'Elephant'})-[:FEEDS_ON]->(f:FoodSource) RETURN f.type"
)

# Ejemplo para Geografía (FOUND_IN)
text2cypher.add_few_shot_example(
    "In which countries or regions are kangaroos found?",
    "MATCH (s:Species {name: 'Kangaroo'})-[:FOUND_IN]->(l:Location) RETURN l.type"
)

# Ejemplo para Bioma/Hábitat (INHABITS)
text2cypher.add_few_shot_example(
    "What kind of ecosystem or biome does the polar bear inhabit?",
    "MATCH (s:Species {name: 'Polar Bear'})-[:INHABITS]->(h:Habitat) RETURN h.type"
)

In [6]:
# ---------------------------------------------------------
# 1. Terminology Maps (English)
# ---------------------------------------------------------

# General
text2cypher.add_terminology_map(
    "animal, creature, species", 
    "Refers to the node with label (:Species)"
)

# Diet distinctions (Carnivores vs Herbivores)
text2cypher.add_terminology_map(
    "carnivore, predator, hunts, preys on, kills", 
    "Use the relationship [:PREYS_ON]->(:Species) to represent hunting and eating other animals."
)
text2cypher.add_terminology_map(
    "herbivore, grazes, eats plants, vegetarian, feeds on", 
    "Use the relationship [:FEEDS_ON]->(:FoodSource) to represent the consumption of non-animal food sources."
)

# Location vs Habitat distinctions
text2cypher.add_terminology_map(
    "country, continent, geographic region, area, located in", 
    "Use the relationship [:FOUND_IN]->(:Location) to refer to specific geographical and political places."
)
text2cypher.add_terminology_map(
    "biome, ecosystem, type of environment, terrain, inhabits", 
    "Use the relationship [:INHABITS]->(:Habitat) to refer to the natural biome or habitat type."
)

text2cypher.add_terminology_map(
    "mentioned in the text, according to the documents, sources say", 
    "Traverse from (:Species)<-[:HAS_ENTITY]-(:Chunk)<-[:HAS_CHUNK]-(:Document)"
)

In [11]:
cypher, results = text2cypher.retrieve("Where are penguins?")

print("Generated Cypher:")
print(cypher)
print("\nResults:")
for result in results:
    print(result)

Generated Cypher:
MATCH (s:Species {name: 'Penguin'})-[:FOUND_IN]->(l:Location) RETURN l.type

Results:
{'l.type': 'South Africa'}
{'l.type': 'Australia'}
{'l.type': 'Antarctica'}
{'l.type': 'New Zealand'}
{'l.type': 'Southern Hemisphere'}
{'l.type': 'Falkland Islands'}
{'l.type': 'Equator'}
{'l.type': 'Northern Hemisphere'}
{'l.type': 'Angola'}
{'l.type': 'Argentina'}
{'l.type': 'Chile'}
{'l.type': 'Namibia'}


## 5. Probar Sistema Agéntico

In [3]:
agentic_rag = AgenticRAG(neo4j)

text2cypher_examples = [
        (
        "How many Mammal species are there?",
        "MATCH (s:Species)-[:BELONGS_TO_CLASS]->(c:AnimalClass {type: 'Mammal'}) RETURN count(s) AS totalMammals"
        ),
        (
        "What is the top speed and maximum weight of a Lion?",
        "MATCH (s:Species {name: 'Lion'}) RETURN s.top_speed_kmh, s.weight_max_kg"
        ),
        (
        "What type of diet do tigers have and what do they prey on?",
        "MATCH (s:Species {name: 'Tiger'})-[:HAS_DIET_TYPE]->(d:DietType), (s)-[:PREYS_ON]->(prey:Species) RETURN d.type, prey.name"
        ),
        (
        "Which animals have a lifespan greater than 50 years?",
        "MATCH (s:Species) WHERE s.lifespan_years > 50 RETURN s.name, s.lifespan_years"
        ),
        (
        "Where do whales migrate to in the winter?",
        "MATCH (s:Species {name: 'Whale'})-[m:MIGRATES_TO {season: 'winter'}]->(l:Location) RETURN l.type"
        ),
        (
        "What does an chimpanzee eat?",
        "MATCH (s:Species {name: 'Chimpanzee'})\
        OPTIONAL MATCH (s)-[:PREYS_ON]->(prey:Species)\
        OPTIONAL MATCH (s)-[:FEEDS_ON]->(food:FoodSource)\
        RETURN s.name AS species,\
        collect(DISTINCT prey.name) AS preys_on,\
        collect(DISTINCT food.type) AS feeds_on"
        ),
        (
        "In which countries or regions are kangaroos found?",
        "MATCH (s:Species {name: 'Kangaroo'})-[:FOUND_IN]->(l:Location) RETURN l.type"
        ),
        (
        "What kind of ecosystem or biome does the polar bear inhabit?",
        "MATCH (s:Species {name: 'Polar Bear'})-[:INHABITS]->(h:Habitat) RETURN h.type"
        )
]

terminology_maps = [
    (
        "animal, creature, species", 
        "Refers to the node with label (:Species)"
    ),
    (
        "what does X eat, what do X eat, diet of X, food of X. When asking what an animal eats, check BOTH relationships: ",
        "[:PREYS_ON]->(:Species) for animal prey AND [:FEEDS_ON]->(:FoodSource) for non-animal food sources. Use OPTIONAL MATCH for both and return all results."
    ),
    (
        "country, continent, geographic region, area, located in", 
        "Use the relationship [:FOUND_IN]->(:Location) to refer to specific geographical and political places."
    ),
    (
        "biome, ecosystem, type of environment, terrain, inhabits", 
        "Use the relationship [:INHABITS]->(:Habitat) to refer to the natural biome or habitat type."
    ),
    (
        "biome, ecosystem, type of environment, terrain, inhabits", 
        "Use the relationship [:INHABITS]->(:Habitat) to refer to the natural biome or habitat type."
    )    
]

for question, cypher in text2cypher_examples:
    agentic_rag.add_text2cypher_example(question, cypher)

for terms, explanation in terminology_maps:
    agentic_rag.add_terminology_map(terms, explanation)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 4462.67it/s]


In [4]:
# Pregunta simple
result = agentic_rag.answer("Give me a complete summary of the lion, including where it lives, what it eats, and its conservation status.")

print("Question:", result['question'])
print("\nAnswer:", result['answer'])
print("\nTool used:", result['iterations'][-1]['retrieval']['tool'])
print("\nIterations:", result['final_critique'])

KeyboardInterrupt: 

In [4]:
# Pregunta simple
result = agentic_rag.answer("How do zebras look like? What does an octopus preys on?")

print("Question:", result['question'])
print("\nAnswer:", result['answer'])
print("\nTools used:", result['iterations'][-1]['retrieval'].get('tools', [result['iterations'][-1]['retrieval'].get('tool')]))
print("\nIterations:", result['final_critique'])

Original question: What does an octopus prey on?
Replacements found: {(13, 20): 'Octopus'}
Normalized question: What does an Octopus prey on?
Question: How do zebras look like? What does an octopus preys on?

Answer: Zebras are heavy-bodied animals with long, slender legs, narrow hooves, and a single toe on each foot [1]. They have black and white stripes that are unique to each individual [1, 3]. Zebras have long necks and heads, and a mane that extends from their forehead along their back to their tail [1]. The Grevy's Zebra is the largest species and has large, rounded ears [4]. Zebras have black skin with white stripes overlaying it [2]. An octopus preys on lobsters [6] and clams [7].

Tools used: ['hybrid_search', 'text2cypher']

Iterations: {'is_complete': True, 'is_faithful': True, 'missing_info': [], 'feedback': 'The answer accurately describes the physical characteristics of zebras and the prey of an octopus, as supported by the provided context.'}


In [4]:
# Pregunta simple
result = agentic_rag.answer("Which critically endangered species live in aquatic environments? " \
"What hunts the African Elephant and what does it hunt? " \
"How are birds socially organized?")

print("Question:", result['question'])
print("\nAnswer:", result['answer'])
print("\nTools used:", result['iterations'][-1]['retrieval'].get('tools', [result['iterations'][-1]['retrieval'].get('tool')]))
print("\nIterations:", result['final_critique'])

KeyboardInterrupt: 

In [8]:
# Pregunta simple
result = agentic_rag.answer("Why does the Toucan have such a large and colorful beak?")

print("Question:", result['question'])
print("\nAnswer:", result['answer'])
print("\nTools used:", result['iterations'][-1]['retrieval'].get('tools', [result['iterations'][-1]['retrieval'].get('tool')]))
print("\nIterations:", result['final_critique'])

Question: Why does the Toucan have such a large and colorful beak?

Answer: This information is not in the knowledge base.

Tools used: ['hybrid_search']

Iterations: {'is_complete': False, 'is_faithful': True, 'missing_info': ['Why does the Toucan have such a large and colorful beak?'], 'feedback': "The answer correctly states that the information is not in the knowledge base. However, the question asks about Toucans, and the provided context is about Puffins, Albatrosses, and Platypuses. Therefore, the answer is incomplete because it does not address the user's question about Toucans."}


In [ ]:
# Pregunta compleja que requiere agregación
result = agentic_rag.answer("How many animals are predators of the Zebrass?")

print("Question:", result['question'])
print("\nAnswer:", result['answer'])
print("\nTools used:", result['iterations'][-1]['retrieval'].get('tools', [result['iterations'][-1]['retrieval'].get('tool')]))
print("\nIterations:", result['final_critique'])

Question: How many animals are predators of the Zebras?

Answer: 6 [1]

Tools used: ['text2cypher']

Iterations: {'is_complete': True, 'is_faithful': True, 'missing_info': [], 'feedback': 'The answer correctly identifies the number of predators of zebras as stated in the context.'}


In [9]:
# Conversación multi-turno
agentic_rag.reset_conversation()

result1 = agentic_rag.answer("Where do iguanas live?")
print("Q1:", result1['question'])
print("A1:", result1['answer'])

result2 = agentic_rag.answer("How do they look?")
print("\nQ2:", result2['question'])
print("A2:", result2['answer'])

Q1: Where do iguanas live?
A1: This information is not in the knowledge base.

Q2: How do they look?
A2: Iguanas vary in color, with some species sporting blue or grey skin [1], and their colors can also vary based on mood, temperature, social status, or health [1]. They have a dewlap under the throat and a dorsal crest on their backs [1]. Their skin tends to be darker in the morning and pales as the day heats [1]. They have varying scales that cover different parts of their bodies [2], and dominant males tend to have darker coloration [1]. Before courtship, some males become bright orange or golden [1].


In [ ]:
neo4j.close()